# Getting Started with ReforesTree

Welcome! This notebook is your first step into the **ReforesTree** dataset, a collection of high-resolution drone imagery from 6 agroforestry sites in Ecuador, paired with field-measured tree parameters and Above Ground Biomass (AGB) estimates.

By the end of this notebook you will know how to:
1. Explore the annotation CSV — species, AGB values, bounding box coordinates
2. Load and display a raw drone image tile
3. Draw bounding boxes over individual tree crowns
4. Use the `AGB_Reforest` dataset class to load everything in one call

No model training here | just data exploration. 

Let's dive into getting familiar with the dataset first.

---
**Dataset path on Kaggle:** `/kaggle/input/reforestree-dataset/`  
**Paper:** [Reiersen et al., 2022](https://arxiv.org/abs/2201.11192)

## 0. Install & Imports

In [ ]:
pip install -U torchgeo

In [ ]:
import os
from pathlib import Path
from typing import Optional, Callable, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.figure import Figure
from PIL import Image

import torch
from torch import Tensor
from torchgeo.datasets import NonGeoDataset
from torchgeo.datasets.errors import DatasetNotFoundError
from torchgeo.datasets.utils import check_integrity, download_and_extract_archive, extract_archive

# dataset root — adjust if running outside Kaggle
ROOT = "/kaggle/input/reforestree-dataset"

print("Imports OK")

---
## 1. Explore the Annotation CSV

The annotation file `mapping/final_dataset.csv` is the backbone of the dataset. It contains one row per labeled tree, linking each tree to:
- Its **bounding box** in the tile image (`xmin`, `ymin`, `xmax`, `ymax`)
- Its **species name** and coarse **group** (banana, cacao, citrus, fruit, timber, other)
- Its **AGB** value in kilograms, computed from field-measured DBH via allometric equations

In [ ]:
df = pd.read_csv(os.path.join(ROOT, "mapping/final_dataset.csv"))

print(f"Total annotated trees: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# How many trees per species group?
group_counts = df['group'].value_counts()
print("Trees per species group:")
print(group_counts.to_string())

In [ ]:
# How many unique fine-grained species names?
print(f"Unique species names: {df['name'].nunique()}")
print(df['name'].value_counts().head(10))

In [ ]:
# AGB distribution summary
print("AGB (kg) statistics:")
print(df['AGB'].describe().round(2))

In [ ]:
# Plot AGB distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: full distribution (right-skewed)
axes[0].hist(df['AGB'], bins=60, color='forestgreen', edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('AGB (kg)', fontsize=12)
axes[0].set_ylabel('Number of trees', fontsize=12)
axes[0].set_title('AGB Distribution (full range)', fontsize=13)
axes[0].grid(axis='y', alpha=0.3)

# Right: zoomed in, excluding top 5% outliers
p95 = df['AGB'].quantile(0.95)
axes[1].hist(df[df['AGB'] <= p95]['AGB'], bins=50, color='steelblue', edgecolor='white', linewidth=0.4)
axes[1].set_xlabel('AGB (kg)', fontsize=12)
axes[1].set_title(f'AGB Distribution (≤ {p95:.0f} kg, 95th percentile)', fontsize=13)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# AGB by species group — boxplot
fig, ax = plt.subplots(figsize=(10, 5))

groups = df['group'].unique()
data_by_group = [df[df['group'] == g]['AGB'].values for g in sorted(groups)]

bp = ax.boxplot(data_by_group, labels=sorted(groups), patch_artist=True, showfliers=False)

colors = ['#4CAF50', '#FF9800', '#795548', '#F44336', '#9C27B0', '#2196F3']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xlabel('Species Group', fontsize=12)
ax.set_ylabel('AGB (kg)', fontsize=12)
ax.set_title('AGB by Species Group (outliers hidden)', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Key observations:**
- AGB is **heavily right-skewed**: most trees are lightweight cacao plants, but a few timber trees dominate the upper tail.
- **Timber** trees have the highest median AGB, followed by **fruit** trees.
- **Banana** trees have near-zero AGB because their biomass is mostly water (not captured by standard allometric equations for woody biomass).

This skew is why we z-score normalize AGB during model training.

---
## 2. Load a Raw Drone Image

The tiles are 4,000 × 4,000 pixel PNG files, organized by site inside `tiles/`. Each filename encodes the site and the pixel coordinates of the tile within the full orthomosaic:

```
<Site Name>_<tile_index>_<x_start>_<y_start>_<x_end>_<y_end>.png
```

At ~2 cm/pixel resolution, each tile covers approximately **80 × 80 metres** of ground.

In [ ]:
# List available sites
tiles_root = Path(ROOT) / "tiles"
sites = sorted([d.name for d in tiles_root.iterdir() if d.is_dir()])
print(f"{len(sites)} sites found:")
for s in sites:
    n = len(list((tiles_root / s).glob("*.png")))
    print(f"  {s}  →  {n} tiles")

In [ ]:
# Pick a specific tile to explore
# Feel free to change the index or site
all_tiles = sorted(list(tiles_root.rglob("*.png")))
print(f"Total tiles: {len(all_tiles)}")

tile_path = all_tiles[25]  # change index to explore other tiles
print(f"\nSelected tile: {tile_path.name}")

In [ ]:
# Load and display the raw image
img = Image.open(tile_path)
print(f"Image size: {img.size}  (width x height)")
print(f"Mode: {img.mode}")

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(img)
ax.axis('off')
ax.set_title(tile_path.name, fontsize=9)
plt.tight_layout()
plt.show()

---
## 3. Draw Bounding Boxes

Now let's overlay the annotated tree bounding boxes on the image. We look up all rows in the CSV where `img_path` matches the filename of our tile.

In [ ]:
# Filter annotations for this tile
tile_df = df[df['img_path'] == tile_path.name].copy()
print(f"Annotated trees in this tile: {len(tile_df)}")
tile_df[['name', 'group', 'AGB', 'xmin', 'ymin', 'xmax', 'ymax']].head(8)

In [ ]:
# Plot image with bounding boxes — colored by species group
GROUP_COLORS = {
    'banana':  '#FFD600',
    'cacao':   '#795548',
    'citrus':  '#FF9800',
    'fruit':   '#E91E63',
    'timber':  '#1565C0',
    'other':   '#9E9E9E',
}

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(img)
ax.axis('off')
ax.set_title(f"{tile_path.name}\n{len(tile_df)} annotated trees", fontsize=10)

for _, row in tile_df.iterrows():
    x1, y1, x2, y2 = row['xmin'], row['ymin'], row['xmax'], row['ymax']
    color = GROUP_COLORS.get(row['group'], '#9E9E9E')
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=1.5, edgecolor=color, facecolor='none'
    )
    ax.add_patch(rect)

# Legend
legend_handles = [
    patches.Patch(facecolor=c, edgecolor=c, label=g)
    for g, c in GROUP_COLORS.items() if g in tile_df['group'].values
]
ax.legend(handles=legend_handles, loc='upper right', fontsize=10,
          framealpha=0.8, title='Species group')

plt.tight_layout()
plt.show()

In [ ]:
# Zoom into a single tree crown — pick the largest AGB tree in this tile
biggest = tile_df.loc[tile_df['AGB'].idxmax()]
x1, y1, x2, y2 = int(biggest.xmin), int(biggest.ymin), int(biggest.xmax), int(biggest.ymax)

# Add some padding around the bbox
pad = 80
crop = img.crop((
    max(0, x1 - pad), max(0, y1 - pad),
    min(img.width, x2 + pad), min(img.height, y2 + pad)
))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: cropped crown
axes[0].imshow(crop)
axes[0].axis('off')
axes[0].set_title(f"Largest tree crown\nSpecies: {biggest['name']}  |  Group: {biggest['group']}")

# Right: same crop with bbox drawn
axes[1].imshow(crop)
# Re-draw bbox relative to the crop origin
ox, oy = max(0, x1 - pad), max(0, y1 - pad)
rect = patches.Rectangle(
    (x1 - ox, y1 - oy), x2 - x1, y2 - y1,
    linewidth=2.5, edgecolor='red', facecolor='none'
)
axes[1].add_patch(rect)
axes[1].text(x1 - ox, y1 - oy - 5,
             f"AGB: {biggest['AGB']:.1f} kg",
             color='white', fontsize=11, fontweight='bold',
             bbox=dict(facecolor='red', alpha=0.8, pad=2))
axes[1].axis('off')
axes[1].set_title('Bounding Box')

plt.tight_layout()
plt.show()

---
## 4. Using the `AGB_Reforest` Dataset Class

Loading images and bounding boxes manually is instructive but verbose. The `AGB_Reforest` class — a subclass of TorchGeo's `NonGeoDataset` — handles all of this for you:

- Scans the `tiles/` directory and builds a sorted list of valid image paths
- Filters out [5 known bad tiles](https://github.com/gyrrei/ReforesTree/issues/6) with GPS mismatches
- Returns each sample as a dictionary of tensors
- Normalizes AGB values (z-score) for model training
- Includes a `plot2()` method for quick visualization

In [ ]:
class AGB_Reforest(NonGeoDataset):
    """ReforesTree dataset — custom subclass of torchgeo.NonGeoDataset."""

    classes = ('other', 'banana', 'cacao', 'citrus', 'fruit', 'timber')
    url = 'https://zenodo.org/records/6813783/files/reforesTree.zip?download=1'
    md5 = 'f6a4a1d8207aeaa5fbab7b21b683a302'
    zipfilename = 'reforesTree.zip'

    def __init__(self, root='data', img_dir='tiles',
                 metadata_path='mapping/final_dataset.csv',
                 transforms=None, download=False, checksum=False):
        self.root = root
        self.img_dir = img_dir
        self.transforms = transforms
        self.checksum = checksum
        self.download = download
        self._verify()

        self.files = self._load_files()
        self.annot_df = pd.read_csv(os.path.join(root, metadata_path))
        self.classes_grp  = self.annot_df['group'].unique()
        self.classes_name = self.annot_df['name'].unique()
        self.class2idx = {c: i for i, c in enumerate(sorted(self.classes_grp))}
        self.name2idx  = {n: i for i, n in enumerate(sorted(self.classes_name))}
        # z-score stats for AGB normalisation
        self.agb_mean = self.annot_df['AGB'].mean()
        self.agb_std  = self.annot_df['AGB'].std()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        filepath = self.files[index]
        image = self._load_image(filepath)
        boxes, labels, agb, labels_name = self._load_target(filepath)
        sample = {'image': image, 'bbox_xyxy': boxes,
                  'label': labels, 'agb': agb, 'label_name': labels_name}
        if self.transforms:
            sample = self.transforms(sample)
        return sample

    def _load_files(self):
        bad = {
            'Carlos Vera Guevara RGB_15_8425_8305_12425_12305.png',
            'Flora Pluas RGB_3_0_11400_4000_15400.png',
            'Flora Pluas RGB_4_0_11578_4000_15578.png',
            'Flora Pluas RGB_23_12782_11400_16782_15400.png',
            'Flora Pluas RGB_24_12782_11578_16782_15578.png',
        }
        return sorted([p for p in Path(self.root, self.img_dir).rglob('*.png')
                       if p.name not in bad])

    def _load_image(self, path):
        with Image.open(path) as img:
            arr = np.array(img, dtype=np.uint8)
        return torch.from_numpy(arr).float().permute(2, 0, 1)  # CxHxW

    def _load_target(self, filepath):
        tile_df = self.annot_df[self.annot_df['img_path'] == filepath.name]
        boxes  = torch.tensor(tile_df[['xmin','ymin','xmax','ymax']].values, dtype=torch.float32)
        labels = torch.tensor([self.class2idx[g] for g in tile_df['group']], dtype=torch.long)
        l_name = torch.tensor([self.name2idx[n]  for n in tile_df['name']],  dtype=torch.long)
        agb    = torch.tensor(
            (tile_df['AGB'].values - self.agb_mean) / self.agb_std, dtype=torch.float32
        )
        return boxes, labels, agb, l_name

    def _verify(self):
        dirs = [os.path.join(self.root, d) for d in ['tiles', 'mapping']]
        if all(os.path.exists(d) for d in dirs):
            return
        if not self.download:
            raise DatasetNotFoundError(self)
        download_and_extract_archive(self.url, self.root,
                                     filename=self.zipfilename,
                                     md5=self.md5 if self.checksum else None)

    def plot2(self, sample, figsize=(12, 12), name_or_group='name', color='crimson'):
        """Plot a tile with labeled bounding boxes."""
        if name_or_group == 'name':
            idx    = sample['label_name'].numpy()
            labels = np.array(self.classes_name)[idx]
        else:
            idx    = sample['label'].numpy()
            labels = np.array(self.classes_grp)[idx]

        image = sample['image'].permute(1, 2, 0).byte().numpy()
        fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
        ax.imshow(image)
        ax.axis('off')

        for i, bbox in enumerate(sample['bbox_xyxy'].numpy()):
            ax.add_patch(patches.Rectangle(
                (bbox[0], bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1],
                linewidth=1.8, edgecolor=color, facecolor='none'
            ))
            ax.text(bbox[0], bbox[1], f"{i}·{labels[i]}",
                    color='white', fontsize=8, fontweight='bold',
                    bbox=dict(facecolor=color, edgecolor='none', pad=1))
        return fig

In [ ]:
# Instantiate the dataset
ds = AGB_Reforest(root=ROOT)

print(f"Number of tiles: {len(ds)}")
print(f"\nFirst 3 tile paths:")
for p in ds.files[:3]:
    print(f"  {p}")

In [ ]:
# Load a sample — returns a dict of tensors
sample = ds[25]  # try different indices: 0, 10, 50, 80 …

print("Sample keys:", list(sample.keys()))
print(f"\n  image      : {sample['image'].shape}    dtype={sample['image'].dtype}")
print(f"  bbox_xyxy  : {sample['bbox_xyxy'].shape}  → {sample['bbox_xyxy'].shape[0]} trees")
print(f"  label      : {sample['label'].shape}     (group index per tree)")
print(f"  label_name : {sample['label_name'].shape} (species index per tree)")
print(f"  agb        : {sample['agb'].shape}       (z-score normalized AGB)")

print(f"\nAGB normalisation stats used: mean={ds.agb_mean:.2f} kg, std={ds.agb_std:.2f} kg")

# De-normalize to see raw kg values
agb_raw = sample['agb'] * ds.agb_std + ds.agb_mean
print(f"\nAGB range in this tile: {agb_raw.min():.1f} – {agb_raw.max():.1f} kg")
print(f"Total AGB in this tile: {agb_raw.sum():.1f} kg")

In [ ]:
# Look up the species names for the trees in this sample
species_names = np.array(ds.classes_name)[sample['label_name'].numpy()]
group_names   = np.array(ds.classes_grp)[sample['label'].numpy()]

print("First 10 trees in this tile:")
for i in range(min(10, len(species_names))):
    print(f"  [{i:02d}]  {species_names[i]:20s}  ({group_names[i]:8s})  "
          f"AGB={agb_raw[i].item():.1f} kg")

In [ ]:
# Visualize the sample with the dataset's built-in plot2() method
# name_or_group='name'  → fine-grained species labels
# name_or_group='group' → coarse group labels (banana, cacao, …)
fig = ds.plot2(sample, figsize=(14, 14), name_or_group='group', color='deepskyblue')
plt.show()

In [ ]:
# Browse a few different tiles side by side
indices = [0, 10, 25, 60]

fig, axes = plt.subplots(2, 2, figsize=(16, 16), constrained_layout=True)

for ax, idx in zip(axes.flat, indices):
    s = ds[idx]
    img_np = s['image'].permute(1, 2, 0).byte().numpy()
    ax.imshow(img_np)
    # draw bboxes
    for bbox in s['bbox_xyxy'].numpy():
        ax.add_patch(patches.Rectangle(
            (bbox[0], bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1],
            linewidth=1.2, edgecolor='red', facecolor='none'
        ))
    n_trees = len(s['bbox_xyxy'])
    agb_total = (s['agb'] * ds.agb_std + ds.agb_mean).sum().item()
    ax.set_title(f"Tile #{idx}  |  {n_trees} trees  |  Total AGB: {agb_total:.0f} kg", fontsize=10)
    ax.axis('off')

plt.show()

---
## 5. Summary

Here's what you've seen in this notebook:

| Step | What we did |
|---|---|
| CSV exploration | Loaded the annotation file, counted trees per species, plotted AGB distribution |
| Raw image | Opened a drone tile with PIL and displayed it |
| Manual bboxes | Drew bounding boxes from the CSV, colored by species group |
| Crown zoom | Cropped and zoomed into the largest individual tree crown |
| Dataset class | Instantiated `AGB_Reforest`, loaded samples as tensors, used `plot2()` |
| Multi-tile browse | Displayed 4 tiles side by side with tree counts and total AGB |

**What's next?**

- Head to the **[Tutorial Notebook](https://www.kaggle.com/code/emanuelgoulart/abg-dl-regressor)** to see how a ResNet-18 + RoI Align + MLP model is built and trained to predict AGB from these images.
- Try changing the tile index in any cell to explore different sites and species compositions.
- Experiment with the AGB de-normalization — can you find the heaviest single tree in the entire dataset?